#Calibración de Matrices Bayesianas con Optuna

Versión interactiva del script `optimizar_matrices_optuna.py`.
Misma lógica, sin necesidad de correr en terminal.

**Flujo:**
1.  Configurar parámetros
2.  Cargar caché precomputado
3.  (Opcional) Evaluar matrices baseline del paper
4.  Ejecutar optimización Optuna
5.  Visualizar resultados
6.  Guardar matrices óptimas

---
##  Celda 1 — Configuración de parámetros
**Modifica aquí los parámetros antes de correr el notebook.**

In [ ]:
# ─────────────────────────────────────────────────────────
#  PARÁMETROS — edita aquí, no toques las celdas de abajo
# ─────────────────────────────────────────────────────────

# Métrica a optimizar:
#   'balanced_accuracy' → igual peso a todas las clases (recomendado si dataset equilibrado)
#   'f1'               → F1-Score macro (más agresivo con clases minoritarias: Metro, Caminar)
METRIC = "balanced_accuracy"

# Número máximo de trials de Optuna
N_TRIALS = 2000

# Early stopping: detener si no hay mejora en N trials consecutivos
# Pon 0 para desactivarlo
PATIENCE = 300

# Semilla para reproducibilidad
SEED = 42

# Timeout en segundos (None = sin límite de tiempo)
TIMEOUT = None

# Comparar con matrices del paper antes de optimizar?
BASELINE_COMPARISON = True

# Persistencia del estudio:
#   None          → en memoria (se pierde al cerrar el notebook)
#   'sqlite:///optuna_calibracion.db'  → guarda en disco, permite reanudar
STORAGE = None
STUDY_NAME = "calibracion_matrices_bayesianas"

# Ruta al .pkl generado por generar_datos_entrenamiento.py
# (None = usa la ruta por defecto en config.GPS_DIR)
CUSTOM_PKL_PATH = None

print(" Parámetros configurados:")
print(f"   Métrica:        {METRIC}")
print(f"   Trials:         {N_TRIALS}")
print(f"   Early stopping: {'desactivado' if PATIENCE == 0 else f'patience={PATIENCE}'}")
print(f"   Seed:           {SEED}")
print(f"   Storage:        {STORAGE or 'en memoria (no persistente)'}")

---
## Celda 2 — Imports y carga de funciones del script

In [ ]:
import sys
import pickle
import time
import json
import datetime
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner, NopPruner

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Agregar project root al path ──────────────────────────────────────────
# El notebook está en pipeline_v4/calibration_and_diagnostics/modal_classification/calibration/bayes/
# El project root está 3 niveles arriba
NOTEBOOK_DIR  = Path(".").resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_v4.src import config

# ── Importar funciones del script de optimización ─────────────────────────
# Así reutilizamos TODA la lógica sin duplicar código
sys.path.insert(0, str(NOTEBOOK_DIR))
from optimizar_matrices_optuna import (
    softmax_rows,
    build_vectorized_cache,
    evaluate_matrices_vectorized,
    build_objective,
    extract_best_matrices,
    print_matrices_report,
    EarlyStoppingCallback,
    update_classifier_matrices,
    MODOS, MODE_TO_IDX,
    BASELINE_CERCANIA, BASELINE_VELOCIDAD,
    BASELINE_DISTANCIA, BASELINE_VELPROM,
)

# ── Ruta al pkl ───────────────────────────────────────────────────────────
PKL_PATH = Path(CUSTOM_PKL_PATH) if CUSTOM_PKL_PATH else (
    config.GPS_DIR / "datos_entrenamiento_optuna.pkl"
)
OUTPUT_JSON = NOTEBOOK_DIR / "matrices_optimas.json"

print(f"Imports correctos")
print(f"   Project root:  {PROJECT_ROOT}")
print(f"   PKL esperado:  {PKL_PATH}")
print(f"   PKL existe:    {' SÍ' if PKL_PATH.exists() else ' NO — corre generar_datos_entrenamiento.py primero'}")

---
##  Celda 3 — Cargar caché y preparar vectorización

In [ ]:
if not PKL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el caché en: {PKL_PATH}\n"
        f"Corre primero: generar_datos_entrenamiento.py"
    )

print(f" Cargando {PKL_PATH.name}...")
with open(PKL_PATH, "rb") as f:
    data_cache = pickle.load(f)

print(f"   {len(data_cache):,} muestras cargadas")

# Distribución de clases
dist = Counter(t["label"] for t in data_cache)
print("\n   Distribución de clases:")
total = sum(dist.values())
for mode in MODOS:
    n = dist.get(mode, 0)
    bar = "" * int(n / total * 40)
    print(f"     {mode:>10}: {n:>5} viajes  {bar}")

# Construir caché vectorizado
print("\n Construyendo caché vectorizado...")
t0 = time.time()
vcache = build_vectorized_cache(data_cache)
n_pts   = len(vcache["idx_c_all"])
n_trips = len(vcache["y_true"])
print(f"   {n_trips:,} viajes válidos | {n_pts:,} puntos GPS totales ({time.time()-t0:.3f}s)")

---
##  Celda 4 — (Opcional) Evaluar matrices baseline del paper

In [ ]:
if BASELINE_COMPARISON:
    print(" Evaluando matrices baseline del paper...")
    base_primary, base_secondary = evaluate_matrices_vectorized(
        vcache,
        BASELINE_CERCANIA, BASELINE_VELOCIDAD,
        BASELINE_DISTANCIA, BASELINE_VELPROM,
        metric=METRIC,
    )
    print_matrices_report(
        BASELINE_CERCANIA, BASELINE_VELOCIDAD,
        BASELINE_DISTANCIA, BASELINE_VELPROM,
        base_primary, base_secondary,
        label="BASELINE (Paper Original)",
        metric=METRIC,
    )
else:
    base_primary, base_secondary = None, None
    print(" Evaluación baseline desactivada (BASELINE_COMPARISON = False)")

---
##  Celda 5 — Ejecutar optimización Optuna
>  Esta celda puede tardar varios minutos dependiendo de `N_TRIALS`.
> La barra de progreso aparece inline.

In [ ]:
sampler = TPESampler(seed=SEED, multivariate=True, group=True)
pruner  = MedianPruner(n_startup_trials=50, n_warmup_steps=0)

load_if_exists = STORAGE is not None
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
    storage=STORAGE,
    load_if_exists=load_if_exists,
)

objective_fn = build_objective(vcache, metric=METRIC)

callbacks = []
if PATIENCE > 0:
    callbacks.append(EarlyStoppingCallback(patience=PATIENCE))

metric_display = "Balanced Accuracy" if METRIC == "balanced_accuracy" else "Macro F1-Score"
print(f"▶ Iniciando optimización ({N_TRIALS} trials | métrica: {metric_display})...\n")

t0 = time.time()
study.optimize(
    objective_fn,
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    n_jobs=1,
    show_progress_bar=True,
    callbacks=callbacks if callbacks else None,
)
elapsed = time.time() - t0

print(f"\n  Completado en {elapsed:.1f}s")
print(f"  Trials ejecutados: {len(study.trials):,}")
print(f"   Mejor loss:        {study.best_value:.6f}")
print(f"   Mejor {metric_display}: {(1 - study.best_value)*100:.2f}%")

---
##  Celda 6 — Visualizar resultados de Optuna

In [ ]:
# Historial de optimización (cómo bajó el loss en cada trial)
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [ ]:
# Importancia de cada parámetro (qué logits impactan más el resultado)
fig = optuna.visualization.plot_param_importances(study)
fig.show()

---
##  Celda 7 — Matrices óptimas y comparación con baseline

In [ ]:
# Reconstruir matrices óptimas
Cercania, Velocidad, Distancia, Velprom = extract_best_matrices(study.best_params)

best_primary, best_secondary = evaluate_matrices_vectorized(
    vcache, Cercania, Velocidad, Distancia, Velprom, metric=METRIC
)

print_matrices_report(
    Cercania, Velocidad, Distancia, Velprom,
    best_primary, best_secondary,
    label="ÓPTIMAS",
    metric=METRIC,
)

if BASELINE_COMPARISON and base_primary is not None:
    primary_name   = "Balanced Accuracy" if METRIC == "balanced_accuracy" else "Macro F1-Score"
    secondary_name = "Macro F1-Score"    if METRIC == "balanced_accuracy" else "Balanced Accuracy"
    print(f"\n  Mejora vs. Baseline del Paper:")
    print(f"     {primary_name}:   {base_primary:.4f} → {best_primary:.4f}  (Δ {best_primary - base_primary:+.4f})")
    print(f"     {secondary_name}: {base_secondary:.4f} → {best_secondary:.4f}  (Δ {best_secondary - base_secondary:+.4f})")

In [ ]:
# Heatmaps de las 4 matrices óptimas (requiere seaborn)
import matplotlib.pyplot as plt
import seaborn as sns

matrices = {
    "Cercanía (3×4)": (Cercania,  ["Cerca Metro", "Cerca Bus", "Sin infraest."]),
    "Velocidad (4×4)": (Velocidad, ["≤6 km/h", "6-20 km/h", "20-80 km/h", ">80 km/h"]),
    "Distancia (5×4)": (Distancia, ["≤1 km", "1-6 km", "6-10 km", "10-18 km", ">18 km"]),
    "Vel. Promedio (2×4)": (Velprom, ["Vprom≤6 km/h", "Vprom>6 km/h"]),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Matrices Bayesianas Calibradas — Valores Óptimos", fontsize=14, fontweight="bold")

for ax, (title, (mat, row_labels)) in zip(axes.flat, matrices.items()):
    sns.heatmap(
        mat,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        xticklabels=MODOS,
        yticklabels=row_labels,
        vmin=0, vmax=1,
        linewidths=0.5,
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Modo de transporte")

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / "matrices_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Guardado: matrices_heatmap.png")

---
## Celda 8 — Guardar matrices óptimas en JSON

In [ ]:
output_data = {
    "study_name":         STUDY_NAME,
    "timestamp":          datetime.datetime.now().isoformat(),
    "n_trials_completed": len(study.trials),
    "best_loss":          float(study.best_value),
    "metric_optimized":   METRIC,
    "balanced_accuracy":  float(best_primary if METRIC == "balanced_accuracy" else best_secondary),
    "macro_f1":           float(best_primary if METRIC == "f1" else best_secondary),
    "patience_used":      PATIENCE,
    "sampler":            "TPESampler (multivariate=True)",
    "matrices": {
        "Cercania":  Cercania.tolist(),
        "Velocidad": Velocidad.tolist(),
        "Distancia": Distancia.tolist(),
        "Velprom":   Velprom.tolist(),
    },
    "bins": {
        "Cercania_rows":  ["Cerca Metro (dist<50m)", "Cerca Bus (dist<50m)", "Sin infraestructura"],
        "Velocidad_rows": ["<=6 km/h", "6-20 km/h", "20-80 km/h", ">80 km/h"],
        "Distancia_rows": ["<=1 km", "1-6 km", "6-10 km", "10-18 km", ">18 km"],
        "Velprom_rows":   ["Vprom<=6 km/h", "Vprom>6 km/h"],
        "columns":        MODOS,
    },
}

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"Matrices guardadas en: {OUTPUT_JSON}")
print(f"   Balanced Accuracy: {output_data['balanced_accuracy']:.4f}")
print(f"   Macro F1-Score:    {output_data['macro_f1']:.4f}")

---
##  Celda 9 — (Opcional) Actualizar bayes_classifier.py con las matrices óptimas
>  **Esto modifica el archivo fuente.** Se crea un backup `.py.bak` automáticamente.

In [ ]:
# Cambia a True para aplicar las matrices al clasificador
APPLY_TO_CLASSIFIER = False

if APPLY_TO_CLASSIFIER:
    print(" Actualizando bayes_classifier.py...")
    success = update_classifier_matrices(Cercania, Velocidad, Distancia, Velprom)
    if not success:
        print(" No se pudo actualizar automáticamente. Copia los valores desde el JSON.")
else:
    print("Para aplicar al clasificador, cambia APPLY_TO_CLASSIFIER = True y re-ejecuta esta celda.")
    print(f"   O carga el JSON desde: {OUTPUT_JSON}")